# Test SOTA TSFM(s) (e.g. Toto) on Gifteval multivariate info vs usiong only univariate. Does multivariate help in practice

### Credit: toto/notebooks/inference_tutorial.ipynb

In [1]:
# --- Step 1: Load bizitobs_l2c/H/short data using datasets package ---
import os
import numpy as np
import pandas as pd
from datasets import load_dataset
from toto.data.util.dataset import MaskedTimeseries
from toto.inference.forecaster import TotoForecaster
from toto.model.toto import Toto
import torch
from datetime import datetime, timedelta

import matplotlib
import matplotlib.pyplot as plt
from collections import defaultdict

/home/roc/01_projects/11_time-series/multivar_timeseries/.venv/lib/python3.11/site-packages/lightning/fabric/__init__.py:40: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


In [2]:
all_data =  [
    "bitbrains_fast_storage/5T",
    "bitbrains_fast_storage/H",
    "bitbrains_rnd/5T",
    "bitbrains_rnd/H",
    # "bizitobs_application/10S", # remove
    "bizitobs_l2c/5T",
    "bizitobs_l2c/H",
    # "bizitobs_service/10S", # remove
    "ett1/15T",
    "ett1/D",
    "ett1/H",
    "ett1/W",
    "ett2/15T",
    "ett2/D",
    "ett2/H",
    "ett2/W",
    "jena_weather/10T",
    "jena_weather/D",
    "jena_weather/H",
]

freq_map = {
            "10S": 10,           # 10 seconds → 10 seconds
            "5T": 5 * 60,        # 5 minutes → 300 seconds
            "10T": 10 * 60,      # 10 minutes → 600 seconds
            "15T": 15 * 60,      # 15 minutes → 900 seconds
            "H": 1 * 3600,       # 1 hour → 3,600 seconds
            "D": 1 * 86400,      # 1 day → 86,400 seconds
            "W": 7 * 86400,      # 1 week → 604,800 seconds
        }

samples_per_batch = 16
forecast_log = {}
metrics_log = []
context_pct = 0.8
prediction_pct = 0.2
context_length_cap = 8_000
prediction_length_cap = 2_000
DEVICE = "cuda"

def load_data(dataset_name, freq):
    # Path to Arrow file directory (datasets expects a directory, not a file)
    DATASET_DIR = f"/home/roc/01_projects/11_time-series/multivar_timeseries/data/gift_eval/{dataset_name}/{freq}"
    # Load the dataset using the datasets package
    ds = load_dataset('arrow', data_files={"train": os.path.join(DATASET_DIR, "data-00000-of-00001.arrow")})
    # Convert to pandas DataFrame for inspection
    df = ds['train'].to_pandas()
    return df


def compute_mae(forecast, target):
    # forecast.samples: shape (1, n_variates, prediction_length, n_samples)
    # target: shape (n_variates, prediction_length)
    pred = np.median(forecast.samples.squeeze().cpu().numpy(), axis=-1)
    mae_per_var = np.mean(np.abs(pred - target), axis=1)
    mae_overall = np.mean(mae_per_var)
    return list(map(lambda x: round(x, 4), mae_per_var)), round(mae_overall, 4)


def plot_forecasts(forecast_log, dataset, freq, mode):
    idx_inputs_start = forecast_log[dataset][freq]['idx_inputs_start']
    idx_inputs_end = forecast_log[dataset][freq]['idx_inputs_end']
    dates = forecast_log[dataset][freq]['dates']
    n_variates = forecast_log[dataset][freq]['n_variates']
    inputs = forecast_log[dataset][freq]['inputs']
    target = forecast_log[dataset][freq]['target']
    forecast = forecast_log[dataset][freq][mode]

    DARK_GREY = "#1c2b34"
    BLUE = "#3598ec"
    PURPLE = "#7463e1"
    LIGHT_PURPLE = "#d7c3ff"
    PINK = "#ff0099"

    matplotlib.rc("axes", edgecolor=DARK_GREY)
    fig = plt.figure(figsize=(12, 6), layout="tight", dpi=150)
    plt.suptitle(f"Toto Forecasts ({dataset}/{freq}) - {mode}")

    for i in range(n_variates):
        # Configure axes
        feature = f"ftr_{i}"
        plt.subplot(n_variates, 1, i + 1)
        if i != 6:
            # only show x tick labels on the bottom subplot
            fig.gca().set_xticklabels([])
        fig.gca().tick_params(axis="x", color=DARK_GREY, labelcolor=DARK_GREY)
        fig.gca().tick_params(axis="y", color=DARK_GREY, labelcolor=DARK_GREY)
        fig.gca().yaxis.set_label_position("right")
        plt.ylabel(feature)
        # plt.xlim(input_df.date.iloc[-960], target_df.date.iloc[-1])
        plt.axvline(dates[idx_inputs_end], color=PINK, linestyle=":")

        # Plot ground truth
        plt.plot(dates[idx_inputs_start: idx_inputs_end], inputs[i], color=BLUE)
        plt.plot(dates[idx_inputs_end:], target[i], color=BLUE)

        # Plot point forecasts
        plt.plot(
            dates[idx_inputs_end:],
            np.median(forecast.samples.squeeze()[i].cpu(), axis=-1),
            color=PURPLE,
            linestyle="--",
        )

        # Plot quantiles
        alpha = 0.05
        qs = forecast.samples.quantile(q=torch.tensor([alpha, 1 - alpha], device=forecast.samples.device), dim=-1)
        plt.fill_between(
            dates[idx_inputs_end:],
            qs[0].squeeze()[i].cpu(),
            qs[1].squeeze()[i].cpu(),
            color=LIGHT_PURPLE,
            alpha=0.8,
        )
        
# print("Columns:", df.columns.tolist())
# print("Shape:", df.shape)
# print(df.head())

# # Inspect one example (first row)
# row = df.iloc[0]
# for col in df.columns:
#     print(f"{col}: type={type(row[col])}, shape={np.array(row[col]).shape if hasattr(row[col], 'shape') else 'N/A'}")

def gen_dates_and_timestamp_seconds(input_series: torch.Tensor, df: pd.DataFrame, freq_map: dict = freq_map) -> torch.Tensor:
    """
    Generate dates for the entire df and timestamp seconds for each time step in input_series.
    """
    # n_var, n_step = input_series.shape
    target_list = df['target'][0]  # List/array of 1D arrays, one per variable
    n_var, n_step = len(target_list), len(target_list[0])
    freq = df['freq'][0].split("-")[0] # To handle cases like "W-THU"
    start_time = df['start'][0]
    # Parse start_time to datetime if needed
    if isinstance(start_time, str):
        start_dt = datetime.fromisoformat(start_time)
    else:
        start_dt = start_time.to_pydatetime() if hasattr(start_time, "to_pydatetime") else start_time
    # Map freq string to seconds
    if freq not in freq_map:
        raise ValueError(f"Unsupported freq: {freq}")
    step_seconds = freq_map[freq]
    # Get start timestamp in seconds since epoch
    start_ts = int(start_dt.timestamp())
    # Generate timestamps for each step
    timestamps = torch.arange(n_step) * step_seconds + start_ts
    dates = pd.to_datetime(timestamps, unit='s')
    n_step_input = input_series.shape[1]
    # Expand to (n_var, n_step_input)
    timestamp_seconds = timestamps[:n_step_input].unsqueeze(0).expand(n_var, n_step_input).to(input_series.device)
    return dates, timestamp_seconds

def main(dataset_name, freq, context_pct, prediction_pct):
    global forecast_log, samples_per_batch, freq_map, DEVICE, metrics_log, context_length_cap, prediction_length_cap
    """    Prepares the input series for Toto from the DataFrame.
    """
    df = load_data(dataset_name, freq)
    # --- Step 2: Prepare target array for Toto (shape: [num_variables, series_length]) ---
    target_list = df['target'][0]  # List/array of 1D arrays, one per variable
    n_variates = len(target_list)
    n_step = len(target_list[0])
    print("total n_step in data:", n_step)
    # print("Type of target_list:", type(target_list))
    # print("Number of variables:", n_variates)
    # print("Shape of first variable:", np.array(target_list[0]).shape)

    # # Stack into 2D array: (num_variables, series_length)
    # target_array = np.stack(target_list, axis=0)
    # print("Final stacked target_array shape:", target_array.shape)


    interval = freq_map[freq]
    context_length = min(int(n_step * context_pct), context_length_cap) # Cap context length for fast experiment
    prediction_length = min(int(n_step * prediction_pct), prediction_length_cap) # Cap prediction length for fast experiment
    idx_inputs_start = -(context_length+prediction_length)
    idx_inputs_end = -prediction_length
    inputs = np.stack([x[idx_inputs_start:idx_inputs_end] for x in target_list])
    target = np.stack([x[idx_inputs_end:] for x in target_list])

    # Debug: Check for NaNs/Infs and shape in inputs
    print("inputs shape:", inputs.shape)
    print("target shape:", target.shape)
    print("inputs dtype:", inputs.dtype)
    print("Any NaNs in inputs?", np.isnan(inputs).any())
    print("Any Infs in inputs?", np.isinf(inputs).any())
    if np.isnan(inputs).any() or np.isinf(inputs).any():
        # Optionally, replace NaNs/Infs with zeros or some other value
        inputs = np.nan_to_num(inputs, nan=0.0, posinf=0.0, neginf=0.0)
        print("NaNs/Infs replaced with 0.0")
    print("\n\n")
    
    input_series = torch.from_numpy(inputs).to(torch.float).to(DEVICE)
    input_series.shape
    dates, timestamp_seconds = gen_dates_and_timestamp_seconds(input_series, df)
    time_interval_seconds = torch.full((n_variates,), interval).to(input_series.device)
    # start_timestamp_seconds = timestamp_seconds[:, 0]
    
    id_mask_multi = torch.zeros_like(input_series)
    id_mask_uni = torch.arange(input_series.shape[0], device=input_series.device).unsqueeze(1).expand_as(input_series)
    
    input_ts_uni = MaskedTimeseries(
        series=input_series,
        # The padding mask should be the same shape as the input series.
        # It should be 0 to indicate padding and 1 to indicate valid values.
        padding_mask=torch.full_like(input_series, True, dtype=torch.bool),
        # The ID mask is used for packing unrelated time series along the Variate dimension.
        # This is used in training, and can also be useful for large-scale batch inference in order to
        # process time series of different numbers of variates using batches of a fixed shape.
        # The ID mask controls the channel-wise attention; variates with different IDs cannot attend to each other.
        # If you're not using packing, just set this to zeros.
        id_mask=id_mask_uni,
        # As mentioned above, these timestamp features are not currently used by the model;
        # however, they are reserved for future releases.
        timestamp_seconds=timestamp_seconds,
        time_interval_seconds=time_interval_seconds,
    )


    input_ts_multi = MaskedTimeseries(
        series=input_series,
        padding_mask=torch.full_like(input_series, True, dtype=torch.bool),
        id_mask=id_mask_multi,
        timestamp_seconds=timestamp_seconds,
        time_interval_seconds=time_interval_seconds,
    )


    toto = Toto.from_pretrained('Datadog/Toto-Open-Base-1.0')
    toto.to(DEVICE)
    # torch.cuda.empty_cache()

    # Optionally enable Torch's JIT compilation to speed up inference. This is mainly
    # helpful if you want to perform repeated inference, as the JIT compilation can
    # take time to wrm up.
    toto.compile()

    forecaster = TotoForecaster(toto.model)
    forecast_uni = forecaster.forecast(
        input_ts_uni,
        # We can set any number of timesteps into the future that we'd like to forecast. Because Toto is an autoregressive model,
        # the inference time will be longer for longer forecasts. 
        prediction_length=prediction_length,
        # TOTOForecaster draws samples from a predicted parametric distribution. The more samples, the more stable and accurate the prediction.
        # This is especially important if you care about accurate prediction intervals in the tails.
        # Toto's evaluations were performed using 256 samples. Set this according to your compute budget.
        num_samples=256,
        # TOTOForecaster also handles batching the samples in order to control memory usage.
        # Set samples_per_batch as high as you can without getting OOMs for maximum performance.
        # If you're doing batch inference, the effective batch size sent to the model is (batch_size x samples_per_batch).
        # In this notebook, we're doing unbatched inference, so the effective batch size is samples_per_batch.
        samples_per_batch=samples_per_batch,
        # KV cache should significantly speed up inference, and in most cases should reduce memory usage too.
        use_kv_cache=True,
    )

    forecast_multi = forecaster.forecast(
        input_ts_multi,
        prediction_length=prediction_length,
        num_samples=256,
        samples_per_batch=samples_per_batch,
        use_kv_cache=True,
    )
    
    if dataset_name not in forecast_log:
        forecast_log[dataset_name] = {}
    forecast_log[dataset_name][freq] = {
        'univariate': forecast_uni,
        'multivariate': forecast_multi,
        'idx_inputs_start': idx_inputs_start,
        'idx_inputs_end': idx_inputs_end,
        'dates': dates,
        'n_variates': n_variates,
        'inputs': inputs,
        'target': target,
    }
    
    mae_uni, mae_uni_overall = compute_mae(forecast_uni, target)
    mae_multi, mae_multi_overall = compute_mae(forecast_multi, target)
    
    features = [f"ftr_{i}" for i in range(n_variates)] + ['overall']
    metrics = pd.DataFrame({
        'dataset': [dataset_name for _ in range(n_variates + 1)],
        'frequency': [freq for _ in range(n_variates + 1)],
        'feature': features,
        'mae_uni': list(mae_uni) + [mae_uni_overall],
        'mae_multi': list(mae_multi) + [mae_multi_overall],
        'context_length': [context_length for _ in range(n_variates + 1)],
        'prediction_length': [prediction_length for _ in range(n_variates + 1)],
    })

    metrics_log.append(metrics)
    return

In [ ]:
from tqdm import tqdm
for data in tqdm(all_data):
    dataset_name, freq = data.split('/')
    print(f"Processing {dataset_name}/{freq}")
    main(dataset_name, freq, context_pct, prediction_pct)

  0%|          | 0/17 [00:00<?, ?it/s]

Processing bitbrains_fast_storage/5T
total n_step in data: 8640
inputs shape: (2, 6912)
target shape: (2, 1728)
inputs dtype: float32
Any NaNs in inputs? True
Any Infs in inputs? False
NaNs/Infs replaced with 0.0





  6%|▌         | 1/17 [00:28<07:35, 28.48s/it]

Processing bitbrains_fast_storage/H
total n_step in data: 721
inputs shape: (2, 576)
target shape: (2, 144)
inputs dtype: float32
Any NaNs in inputs? False
Any Infs in inputs? False





 12%|█▏        | 2/17 [00:33<03:40, 14.70s/it]

Processing bitbrains_rnd/5T
total n_step in data: 8640
inputs shape: (2, 6912)
target shape: (2, 1728)
inputs dtype: float32
Any NaNs in inputs? True
Any Infs in inputs? False
NaNs/Infs replaced with 0.0





In [ ]:
metrics_combine = pd.concat(metrics_log, ignore_index=True)

metrics_combine.to_csv(f"metrics_{context_pct}_{prediction_pct}.csv", index=False)

In [ ]:
metrics_combine

,dataset,frequency,feature,mae_uni,mae_multi,context_length,prediction_length
0,bitbrains_fast_storage,5T,ftr_0,NaN,NaN,6912,1728
1,bitbrains_fast_storage,5T,ftr_1,NaN,NaN,6912,1728
2,bitbrains_fast_storage,5T,overall,NaN,NaN,6912,1728
3,bitbrains_fast_storage,H,ftr_0,23.526199,23.541800,576,144
4,bitbrains_fast_storage,H,ftr_1,0.451100,0.453500,576,144
...,...,...,...,...,...,...,...
87,ett2,W,ftr_3,4115.142578,3796.415283,82,20
88,ett2,W,ftr_4,4903.581543,4940.157715,82,20
89,ett2,W,ftr_5,717.176697,751.770081,82,20
90,ett2,W,ftr_6,13873.790039,12343.799805,82,20


In [ ]:
metrics_combine.query("feature == 'overall'").to_csv(f"metrics_{context_pct}_{prediction_pct}_overall.csv", index=False)